# Detect Columns by using ML

---

In [19]:
import numpy as np
import pandas as pd
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functions import *
import scipy
import signal
import io
import cv2
import os
import glob
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

**Machine learning (ML)** is a branch of artificial intelligence. It allows a computer to learn from data and to improve decision making with experience.

---

## Using Random Forest

**L'Arbre de Décision (Decision Tree) :**
Imagine un jeu de "Qui est-ce ?". L'algorithme pose une série de questions par oui/non sur les caractéristiques (features) de tes données pour arriver à une conclusion. Par exemple : "La variance de cette colonne est-elle supérieure à 450 ?" -> Si oui, on va à droite ; si non, on va à gauche.

Le Random Forest repose sur l'apprentissage d'ensemble (Ensemble Learning), et plus précisément sur une technique appelée Bagging (Bootstrap Aggregating). L'algorithme va créer une "forêt" composée de dizaines, voire de centaines d'arbres de décision.

Pour classer une nouvelle colonne (Normale vs Défectueuse), la forêt fait passer les données de la colonne dans tous ses arbres. Chaque arbre vote. La classe qui obtient la majorité des votes l'emporte.

In [20]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path)
            images.append(img)
            
    return images

### Features that we use : 

- Moyenne `mean` : Valeur moyenne des intensités de la colonne. Luminosité globale de la colonne. Si trop ou trop basse peut indiquer anomalie.
- Ecart-type `std` : Dispersion autour de la moyenne. Colonne peut-être *noisy* si l'écart-type est très élevé.
- Min et Max : Si la colonne à des valeurs super hautes ou super basses c'est que l'amplificateur est défectueux (dixit Phlypo)
- Range `range` : Différence entre max-min
- Médian `median` : Valeur centrale. Comparaison avec la moyenne intéressant.
- Quantile `q25` `q75` : Quartile.
- Skewness `skewness` : Asymétrie 
- Kurtosis `kurtosis`: Applatissement 
- Energie `energy` : Energie d'un signal (dixit Signals and Systems)
- Entropie `entropy` : J'ai pas vrmt compris / Mesure du désordre ou de l'incertitude dans la distribution des intensités.
- Nombres de pics `num_peaks` : les pics dans la colonne si gros peut-être *fragmented*
- Moyenne du gradiant `mean_gradient` : Moyenne des différences entre pixels consécutifs

#### A faire : 
- Différence Spatiale (un peu comme mes autres méthodes)
- Différence Temporelle (pour les blinking)

In [30]:
def extract_column_features(img_prev, img_curr, img_next, x):
    # Forcer en float64
    col_prev = img_prev[:, x].flatten().astype(np.float64)
    col_curr = img_curr[:, x].flatten().astype(np.float64)
    col_next = img_next[:, x].flatten().astype(np.float64)
    
    height, width = img_curr.shape[:2]
    col_mean = np.mean(col_curr)
    
    # =========================================================================
    # 1. MÉTHODE OPTIMISÉE : LT_moy (Local Threshold Moyenne)
    # Ton opti globale montre qu'une fenêtre autour de 25-30 est idéale en moyenne.
    # On va calculer l'écart absolu entre la colonne et sa "tendance locale".
    # =========================================================================
    taille_fenetre_opti = 25 
    x_min = max(0, x - (taille_fenetre_opti // 2))
    x_max = min(width, x + (taille_fenetre_opti // 2) + 1)
    
    tendance_locale = np.mean(img_curr[:, x_min:x_max])
    std_locale = np.std(img_curr[:, x_min:x_max])
    
    # Feature : Distance à la tendance locale (Le RF trouvera le fameux facteur_std tout seul !)
    ecart_lt_moy = abs(col_mean - tendance_locale)
    
    # Feature "Normalisée" : Combien de fois l'écart-type local cette colonne représente-t-elle ?
    # C'est exactement ce que ton facteur_std calculait !
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5) 
    
    # =========================================================================
    # 2. MÉTHODE OPTIMISÉE : LT_band_moy (Analyse par Bandes)
    # Tes tests montrent qu'une bande fait idéalement ~120-150 pixels de haut.
    # On rend ça dynamique : nb_bandes s'adapte à la résolution !
    # =========================================================================
    hauteur_bande_cible = 135 # La moyenne magique issue de tes optis (VGA/4, SXGA/7, HD/8)
    num_bandes = max(1, height // hauteur_bande_cible)
    hauteur_reelle = height // num_bandes
    
    max_ecart_bande = 0
    max_ratio_bande = 0
    
    # On cherche la PIRE bande de cette colonne
    for b in range(num_bandes):
        y_start = b * hauteur_reelle
        y_end = height if b == num_bandes - 1 else (b + 1) * hauteur_reelle
        
        # Moyenne du segment de la colonne
        segment_col = np.mean(col_curr[y_start:y_end])
        # Tendance locale sur cette même bande (fenêtre spatiale)
        tendance_bande = np.mean(img_curr[y_start:y_end, x_min:x_max])
        std_bande = np.std(img_curr[y_start:y_end, x_min:x_max])
        
        ecart = abs(segment_col - tendance_bande)
        ratio = ecart / (std_bande + 1e-5)
        
        # On garde la pire anomalie trouvée sur les bandes (Idéal pour défauts fragmentés)
        if ecart > max_ecart_bande:
            max_ecart_bande = ecart
        if ratio > max_ratio_bande:
            max_ratio_bande = ratio
            
    # =========================================================================
    # 3. Le Reste de nos meilleures features (Temporelles et Spatiales)
    # =========================================================================
    if x == 0:
        neighbor_mean = np.mean(img_curr[:, x+1])
    elif x == width - 1:
        neighbor_mean = np.mean(img_curr[:, x-1])
    else:
        neighbor_mean = (np.mean(img_curr[:, x-1]) + np.mean(img_curr[:, x+1])) / 2.0
    spatial_diff = abs(col_mean - neighbor_mean)
    
    mean_prev = np.mean(col_prev)
    mean_next = np.mean(col_next)
    diff_temp_absolue = abs(col_mean - mean_prev)
    scintillement_temporel = abs(col_mean - ((mean_prev + mean_next) / 2.0))

    return {
        # Les Features de tes méthodes optimisées :
        'ecart_lt_moy': ecart_lt_moy,
        'ratio_lt_moy': ratio_lt_moy, # Indique le "facteur_std" de cette colonne
        'max_ecart_bande': max_ecart_bande, # Résultat du LT_band_moy
        'max_ratio_bande': max_ratio_bande,
        
        # Nos classiques :
        'spatial_diff': spatial_diff,
        'diff_temp_absolue': diff_temp_absolue,
        'scintillement_temporel': scintillement_temporel,
        
        # Statistiques globales de base :
        'mean': col_mean,
        'energy': np.sum(col_curr ** 2) / len(col_curr)
    }

In [28]:
def process_single_image(img_num, images_list, json_data):
    """Fonction intermédiaire exécutée par chaque cœur du processeur"""
    X_img = []
    y_img = []
    defects = get_defect_coordinates(json_data, img_num)

    img_curr = images_list[img_num]
    img_prev = images_list[img_num - 1] if img_num > 0 else img_curr
    img_next = images_list[img_num + 1] if img_num < (len(images_list) - 1) else img_curr
    
    for x in range(img_curr.shape[1]):
        # On passe maintenant le trio temporel à la fonction
        features = extract_column_features(img_prev, img_curr, img_next, x)
        
        if x in defects:
            label = 1
        else:
            label = 0
            
        X_img.append(list(features.values()))
        y_img.append(label)
        
    return X_img, y_img

def build_dataset(images, json_data):
    print(f"Lancement de l'extraction sur {len(images)} images en parallèle...")
    
    # n_jobs=-1 (utilisation de tous les coeurs)
    # return_as="generator" permet à tqdm de se mettre à jour en temps réel
    result_generator = Parallel(n_jobs=-1, return_as="generator")(
        delayed(process_single_image)(img_num, images, json_data) 
        for img_num in range(len(images))
    )
    
    X = []
    y = []
    
    # On enveloppe le générateur avec tqdm pour la barre de progression
    for X_img, y_img in tqdm(result_generator, total=len(images), desc="Extraction Multicoeur"):
        X.extend(X_img)
        y.extend(y_img)
        
    return np.array(X), np.array(y)

### Entraînement

In [31]:
X, y = build_dataset(load_images(type='HD', sequence='sequence_2', dyn='low dyn with columns 3'), load_json('results/HD_sequence_2_config_3.json'))

Lancement de l'extraction sur 300 images en parallèle...


Extraction Multicoeur:   0%|          | 0/300 [00:00<?, ?it/s]

In [32]:
# --- ÉTAPE DE RÉÉQUILIBRAGE (UNDERSAMPLING) ---
# On récupère les indices des défauts et des colonnes saines
indices_defauts = np.where(y == 1)[0]
indices_sains = np.where(y == 0)[0]

# On choisit aléatoirement autant de colonnes saines qu'il y a de défauts
# (Tu peux multiplier len(indices_defauts) par 3 ou 4 si tu veux un peu plus de saines)
nb_echantillons = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_echantillons, replace=False)

# On rassemble les indices et on filtre X et y
indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_balanced = X[indices_finaux]
y_balanced = y[indices_finaux]

print(f"Nouveau Dataset : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} colonnes saines.")

# --- ENTRAÎNEMENT DU MODÈLE ---
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

Nouveau Dataset : 5064 défauts et 10128 colonnes saines.
              precision    recall  f1-score   support

           0       0.83      0.93      0.88      2010
           1       0.82      0.63      0.71      1029

    accuracy                           0.83      3039
   macro avg       0.83      0.78      0.79      3039
weighted avg       0.83      0.83      0.82      3039



In [25]:
# Liste des noms de tes features dans le même ordre que ton dictionnaire
feature_names = ['spatial difference', 'mean', 'num_peaks', 'std', 'energy', 'ecart_tendance_50',
                'diff_temp_absolue', 'scintillement_temporel']

importances = clf.feature_importances_
for name, importance in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True):
    print(f"{name}: {importance:.3f}")

ecart_tendance_50: 0.316
spatial difference: 0.225
num_peaks: 0.092
std: 0.087
scintillement_temporel: 0.086
diff_temp_absolue: 0.073
energy: 0.061
mean: 0.061


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# =========================================================================
# 1. PARAMÈTRES DU SUPER DATASET
# =========================================================================
CAMERA_TYPE = 'VGA'  # Change ceci en 'HD' ou 'SXGA' selon ce que tu veux entraîner

# Mapping pour faire correspondre le numéro de config au nom du dossier 'dyn'
dyn_mapping = {
    1: 'low dyn with columns 1', # Blinking
    2: 'low dyn with columns 2', # Noisy
    3: 'low dyn with columns 3'  # Noisy Blinking
}

X_list = []
y_list = []

# =========================================================================
# 2. BOUCLE D'EXTRACTION AUTOMATIQUE (Les 9 combinaisons)
# =========================================================================
for seq in [1, 2, 3]:
    for config in [1, 2, 3]:
        print(f"\n🚀 Lancement : {CAMERA_TYPE} | Séquence {seq} | Configuration {config}")
        
        sequence_name = f'sequence_{seq}'
        dyn_name = dyn_mapping[config]
        
        # Adapte le chemin vers ton dossier results si besoin (ex: '../results/...')
        json_path = f'results/{CAMERA_TYPE}_{sequence_name}_config_{config}.json' 
        
        try:
            # Chargement des données
            images = load_images(type=CAMERA_TYPE, sequence=sequence_name, dyn=dyn_name)
            json_data = load_json(json_path)
            
            # Extraction multicœur (ta fonction existante va s'occuper de tout)
            X_batch, y_batch = build_dataset(images, json_data)
            
            X_list.append(X_batch)
            y_list.append(y_batch)
            
        except Exception as e:
            print(f"⚠️ Erreur ou fichier manquant pour Seq {seq} / Config {config} : {e}")
            continue # On passe au suivant si un dossier n'existe pas

# Fusion de tous les batchs en de grands tableaux Numpy
X_global = np.vstack(X_list)
y_global = np.concatenate(y_list)

print(f"\n✅ Extraction terminée ! Taille totale du dataset : {X_global.shape[0]} colonnes.")

# =========================================================================
# 3. RÉÉQUILIBRAGE DU SUPER DATASET (Très important vu la taille)
# =========================================================================
indices_defauts = np.where(y_global == 1)[0]
indices_sains = np.where(y_global == 0)[0]

print(f"Avant équilibrage : {len(indices_defauts)} défauts et {len(indices_sains)} colonnes saines.")

# On prend 2 fois plus de colonnes saines que de défauts pour un bon apprentissage
nb_sains_a_garder = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_sains_a_garder, replace=False)

indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_balanced = X_global[indices_finaux]
y_balanced = y_global[indices_finaux]

print(f"Après équilibrage : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} colonnes saines.")

# =========================================================================
# 4. ENTRAÎNEMENT DU MODÈLE FINAL
# =========================================================================
# Optionnel : Transformer en DataFrame pour garder les noms de tes features
noms_colonnes = list(extract_column_features(images[0], images[0], images[0], 0).keys())
X_balanced_df = pd.DataFrame(X_balanced, columns=noms_colonnes)

X_train, X_test, y_train, y_test = train_test_split(X_balanced_df, y_balanced, test_size=0.2, random_state=42)

print("\n🌲 Entraînement du Random Forest en cours...")
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n📊 RÉSULTATS DU MODÈLE GLOBAL :")
print(classification_report(y_test, y_pred))

# Affichage du Top 5 des features
print("\n🏆 TOP 5 DES CARACTÉRISTIQUES LES PLUS IMPORTANTES :")
importances = clf.feature_importances_
for name, importance in sorted(zip(noms_colonnes, importances), key=lambda x: x[1], reverse=True)[:5]:
    print(f"{name}: {importance:.3f}")


🚀 Lancement : VGA | Séquence 1 | Configuration 1
Lancement de l'extraction sur 562 images en parallèle...


Extraction Multicoeur:   0%|          | 0/562 [00:00<?, ?it/s]


🚀 Lancement : VGA | Séquence 1 | Configuration 2
Lancement de l'extraction sur 562 images en parallèle...


Extraction Multicoeur:   0%|          | 0/562 [00:00<?, ?it/s]


🚀 Lancement : VGA | Séquence 1 | Configuration 3
Lancement de l'extraction sur 562 images en parallèle...


Extraction Multicoeur:   0%|          | 0/562 [00:00<?, ?it/s]


🚀 Lancement : VGA | Séquence 2 | Configuration 1
Lancement de l'extraction sur 927 images en parallèle...


Extraction Multicoeur:   0%|          | 0/927 [00:00<?, ?it/s]

KeyboardInterrupt: 